# Day 5 — Spark SQL and MLlib

All data are synthetic. Run from top to bottom. Work in pairs and pause after each result to explain its meaning. Use TEACHING_GUIDE.md and TASK_CARDS.md for timing. Optional sections are marked. Numerical outputs are examples, not evidence about real operations.

## 1. Start local Spark

Requires Java 17 and PySpark 4.0.1. This standalone lab runs on one laptop using two local threads. Do not present its timing as a cluster benchmark. Use OFFLINE_ACTIVITY.md if the environment was not successfully rehearsed.

In [ ]:
import os
os.environ.setdefault('SPARK_LOCAL_IP','127.0.0.1')
from pyspark.sql import SparkSession,functions as F
spark=SparkSession.builder.master('local[2]').appName('course-day5').config('spark.ui.enabled','false').config('spark.sql.shuffle.partitions','2').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:',spark.version)
df=spark.range(0,10000).withColumn('region',F.when(F.col('id')%2==0,'A').otherwise('B')).withColumn('amount',(F.col('id')%100+1).cast('double'))
print('Rows:',df.count())
df.show(5)

## 2. Aggregate with DataFrames and SQL

Transformations build a plan; actions trigger execution. groupBy may require shuffling records by key. Only collect a tiny aggregate, not a large source table.

In [ ]:
totals=df.groupBy('region').agg(F.count('*').alias('orders'),F.sum('amount').alias('revenue')).orderBy('region')
totals.show()
df.createOrReplaceTempView('orders')
sql_totals=spark.sql('SELECT region, COUNT(*) AS orders, SUM(amount) AS revenue FROM orders GROUP BY region ORDER BY region')
assert totals.collect()==sql_totals.collect()
print('DataFrame and SQL totals agree.')
totals.explain()

## 3. Train a Spark MLlib regression pipeline

Synthetic route features and target are constructed in Spark. The id-based split is predetermined for this independent synthetic example, not a recommended temporal split. Real chronology or grouped identities require a suitable split. VectorAssembler creates the feature vector expected by MLlib.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
routes=spark.range(0,1000).withColumn('distance',1+30*F.rand(seed=42)).withColumn('stops',F.floor(1+7*F.rand(seed=43))).withColumn('duration',12+2.4*F.col('distance')+4*F.col('stops')+3*F.randn(seed=44)).cache()
train=routes.filter(F.col('id')%5!=0)
test=routes.filter(F.col('id')%5==0)
pipeline=Pipeline(stages=[VectorAssembler(inputCols=['distance','stops'],outputCol='features'),LinearRegression(featuresCol='features',labelCol='duration',regParam=.01,maxIter=50)])
model=pipeline.fit(train)
predictions=model.transform(test)
evaluator=RegressionEvaluator(labelCol='duration',predictionCol='prediction',metricName='mae')
mae=evaluator.evaluate(predictions)
training_mean=train.agg(F.avg('duration')).first()[0]
baseline_mae=test.select(F.avg(F.abs(F.col('duration')-F.lit(training_mean)))).first()[0]
print('Training rows:',train.count(),'Test rows:',test.count())
print('Model MAE:',round(mae,3),'Baseline MAE:',round(baseline_mae,3))
assert mae<baseline_mae
predictions.select('duration','prediction').show(5)
routes.unpersist()
spark.stop()

# Your pair challenge

Use the working examples above. Complete the tasks below in new cells. For model experiments use training/validation data; do not optimise against a final test set.

Change the SQL to compute mean amount by region. Compare it with revenue divided by order count.

In [ ]:
# Add your experiment or calculation here.

Explain the difference between a transformation and an action using this notebook.

In [ ]:
# Add your experiment or calculation here.